In [1]:
#l2
import numpy as np
import math

n_param = 0.51
delta = 0.99
nn = (n_param + 0.5) / 2.0
dd = (delta + 1.0) / 2.0

def int64L2(B):
    B = np.array(B, dtype=object)
    d, n = B.shape
    G = np.zeros((n, n), dtype=object)

    for j in range(n):
        for i in range(j + 1):
            s = sum(B[l, j] * B[l, i] for l in range(d))
            G[j, i] = G[i, j] = s

    rr = np.zeros((n, n), dtype=float)
    uu = np.zeros((n, n), dtype=float)
    rr[0, 0] = float(G[0, 0])
    k = 1
    while k < n:
        for i in range(k + 1):
            rr[i, k] = float(G[i, k])
            for j in range(i):
                rr[i, k] -= rr[j, k] * uu[j, i]
            uu[i, k] = rr[i, k] / rr[i, i]
        max_val = max(abs(uu[j, k]) for j in range(k))
        if max_val > nn:
            for j in range(k - 1, -1, -1):
                X = math.floor(float(uu[j, k]) + 0.5)
                if X:
                    for i in range(d):
                        B[i, k] -= X * B[i, j]
                    for i in range(n):
                        dot = sum(B[l, i] * B[l, k] for l in range(d))
                        G[k, i] = G[i, k] = dot
                    for i in range(j):
                        uu[i, k] -= X * uu[i, j]
            continue
        if dd * rr[k-1, k-1] < rr[k, k] + uu[k-1, k]**2 * rr[k-1, k-1]:
            k += 1
        else:
            B[:, [k-1, k]] = B[:, [k, k-1]]
            for m in [k-1, k]:
                for i in range(n):
                    dot = sum(B[l, i] * B[l, m] for l in range(d))
                    G[m, i] = G[i, m] = dot
            for col in [k-1, k]:
                for i in range(col + 1):
                    rr[i, col] = float(G[i, col])
                    for j in range(i):
                        rr[i, col] -= rr[j, col] * uu[j, i]
                    uu[i, col] = rr[i, col] / rr[i, i]
            k -= 1
            if k < 1:
                k = 1
    return np.array(B, dtype=np.int64)


def basis_norm(B):
    return sum(int(x)**2 for x in B.flatten())


B = np.array([
    [105,  42,  77],
    [ 34,  19,  61],
    [  8,  11,  23]
], dtype=np.int64)

before = basis_norm(B)
R = int64L2(B)
after = basis_norm(R)

print("Original basis:")
print(B)

print("\nReduced basis:")
print(R)

print("\nSquared norm before:", before)
print("Squared norm after :", after)

print("\nReduction happened:", after < before)

Original basis:
[[105  42  77]
 [ 34  19  61]
 [  8  11  23]]

Reduced basis:
[[ -7  21  28]
 [ 23  -4   0]
 [  1 -14  24]]

Squared norm before: 24670
Squared norm after : 2592

Reduction happened: True
